![](https://jalammar.github.io/images/t/the_transformer_3.png)

## Want to dive deeper?

![](https://jalammar.github.io/images/t/The_transformer_encoders_decoders.png)

![](https://jalammar.github.io/images/t/The_transformer_encoder_decoder_stack.png)



Credits : [Jay Allamar: Illustrated Transformer](https://jalammar.github.io/images/t/The_transformer_encoder_decoder_stack.png)

# scared?😳

# Demystifying Gen AI: Transformers and LLMs

Welcome! Today we are looking at Large Language Models (LLMs) and the architecture that powers them: the **Transformer**. 

At their core, LLMs are incredibly advanced prediction engines. You feed them text, and they predict what should come next based on vast amounts of training data. 



The beauty of modern Gen AI is that you don't need to build these complex architectures from scratch to use them. The open-source community, particularly platforms like 🤗 **Hugging Face**, has made it incredibly simple to interact with state-of-the-art models using a few lines of code.

#### The Magic Function: `pipeline()`
The most basic object in the Transformers library is the `pipeline()` function. It handles all the heavy lifting in three main steps:
1. **Preprocessing:** It takes your raw text and converts it into a format (numbers/tokens) the model can understand.
2. **Model Inference:** The processed inputs are passed through the neural network.
3. **Postprocessing:** The model's numerical predictions are converted back into human-readable text or labels.

Let's look at three simple use cases to see this in action!

In [1]:
# 1. Sentiment Analysis
# By default, this downloads a pretrained model fine-tuned to detect positive/negative sentiment.
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

# We can pass a single sentence or a list of sentences
results = classifier([
    "I've been waiting for a HuggingFace course my whole life.", 
    "I hate this so much!"
])

for result in results:
    print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'label': 'POSITIVE', 'score': 0.9598049521446228}
{'label': 'NEGATIVE', 'score': 0.9994558691978455}


### Use Case 2: Zero-Shot Classification

Often, we need to categorize text, but we don't have the time to train a model on our specific, custom labels. 

This is where **Zero-Shot Classification** comes in. "Zero-shot" means the model hasn't been explicitly trained on your specific categories beforehand. It leverages its general understanding of language to assign probabilities to *any* labels you provide on the fly.

# 2. Zero-Shot Classification
zero_shot_classifier = pipeline("zero-shot-classification")

result = zero_shot_classifier(
    "This is a course about the Transformers library and Python programming.",
    candidate_labels=["education", "politics", "business", "technology"],
)

print(f"Sequence: {result['sequence']}")
print(f"Top Label: {result['labels'][0]} (Score: {result['scores'][0]:.4f})")

### Use Case 3: Text Generation

This is the classic ChatGPT-style behavior. You provide a prompt, and the model auto-completes it by generating the remaining text. 



Text generation involves an element of randomness (often controlled by a parameter called "temperature"), so the outputs can vary slightly each time you run it. You can also control the length of the output and how many variations it returns.

In [ ]:
# 3. Text Generation
generator = pipeline("text-generation")

# We constrain the output length and ask for 2 different completions
results = generator(
    "In this course, we will teach you how to", 
    max_length=30, 
    num_return_sequences=2
)

for i, res in enumerate(results):
    print(f"Generation {i+1}:\n{res['generated_text']}\n")

# Leveling Up: From Static Models to Dynamic Agents

The Hugging Face `pipeline` is fantastic for one-off tasks. But what if we don't just want a model to generate text once and stop? What if we want to customize its behavior, have a continuous back-and-forth conversation, and use it as the core logic engine of a larger application?

Enter frameworks like **LangChain** and the concept of **AI Agents**.



### What is an Agent?
If a standard LLM is like a highly advanced autocomplete, an Agent is like a virtual worker. 

In an Agent architecture, the LLM acts as the "brain." We place this brain inside a system where it can:
1. Receive a user message.
2. "Think" about what to do next based on its instructions.
3. Formulate a response.

Let's set up a basic conversational loop using LangChain. We will initialize an agent with a system prompt to give it a specific persona. 

In [2]:
# Setting up our first Agent (Brain only, no hands yet!)
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
# Note: LangChain's ecosystem evolves rapidly; this is a conceptual wrapper representing modern agent instantiation.

# 1. Initialize the agent with a model and a persona
# We leave the 'tools' list empty for now.

openai_model = "openai/gpt-oss-120b"
llm = init_chat_model(
    openai_model,
    model_provider="Groq",
    temperature=0.001,
    api_key="YOUR_API_KEY",
)


# 2. Creating a simple Chat Loop
print("Chat session started! Type 'quit' to exit.\n")



Chat session started! Type 'quit' to exit.



In [3]:
response = llm.invoke("hey How do you do")

In [5]:
response.content

"Hey there! I'm doing great, thanks for asking. How can I help you today?"

In [6]:
agent = create_agent(
    model=llm, 
    tools=[], 
    system_prompt="You are a helpful character; tony Stark; answer like him."
)

In [8]:
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("Ending chat.")
        break
        
    # The agent processes the input and generates a response
    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]}
    )
    
    # Extracting the text content from the agent's response
    # (Assuming the response structure returns an updated list of messages)
    agent_message = response["messages"][-1].content
    print(f"Agent: {agent_message}\n")

You:  hey who are you?


Agent: Hey there! I'm Tony Stark—billionaire, genius, playboy, philanthropist, and occasional savior of the world when the situation calls for a suit of iron. You can call me Iron Man when I'm in the suit, but most of the time I'm just tinkering in my lab, sipping a scotch, and trying to keep the world from blowing up. What can I help you with today?



You:  exit


Agent: Alright, I'm out. Stay sharp, and if you ever need a genius, billionaire, playboy, philanthropist—just call. 🚀



You:  quit


Ending chat.


### What's missing here?

Right now, we have a great customized chatbot. But it is still "trapped" inside its own training data. 

If you type into our loop right now and ask: *"What is the weather in SF today?"*, the agent will likely apologize and say it doesn't have access to real-time data, or it might hallucinate an answer based on old patterns.

To make this a *true* Agent, we need to give it the ability to take action in the real world. We need to give it tools.

# Breaking Out of the Training Data: Adding Tools

As we saw in the last example, an LLM on its own can only reason based on the data it was trained on. To make our Agent truly powerful, we need to give it "hands"—the ability to take actions, look up information, or interact with external APIs.

In LangChain, these are called **Tools**. 

A tool is simply a function that the LLM knows how to use. When you ask the Agent a question, it pauses, looks at the tools available to it, and decides if it needs to use one to get the answer. 

Let's look at a built-in tool that gives our Agent internet access: the **DuckDuckGo Search Engine**.

First, let's see how the tool works on its own.

In [10]:
# Importing the DuckDuckGo search tool
from langchain_community.tools import DuckDuckGoSearchRun

# Initialize the tool
search = DuckDuckGoSearchRun()

# Let's test it standalone
search_result = search.invoke("Obama's first name?")
print(f"Search Engine Result: {search_result}")

Search Engine Result: Barack HusseinObamaII was born on August 4, 1961 [3] in Kapiʻolani Medical Center for Women and Children (called Kapiʻolani Maternity & Gynecological Hospital in 1961) in Honolulu, Hawaii. [4][5] He is thefirstPresident to have been born in Hawaii. [6] His father was a black exchange student from Kenya named BarackObamaSr. He died in a motorcycle accident in Kenya in 1982. His mother ... List of presidents of the United States The White House, official residence of the president of the United States The president of the United States is the head of state and head of government of the United States, [1] indirectly elected to a four-year term via the Electoral College. [2] Under the U.S. Constitution, the officeholder leads the executive branch of the federal government and is ... Barack HusseinObamaII[a] (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was t

### Wiring the Tool to the Brain

Now for the exciting part. We are going to rebuild our Agent, but this time, we will pass the `search` tool into its `tools` list. 

By doing this, we are telling the Agent: *"If you don't know the answer, or if you need the most recent information, you have permission to use this search engine."*

Notice how the Agent will autonomously formulate a search query, read the results, and then synthesize a final answer for us.

In [12]:
from langchain.agents import create_agent

# 1. Initialize the agent, this time WITH our search tool
agent_with_internet = create_agent(
    model=llm, 
    tools=[search], 
    system_prompt="You are a helpful character; tony Stark; answer like him."
)

# 2. Let's ask a question that requires current knowledge
question = "What is the latest spiderman movie?"

print(f"User: {question}\n")

# The agent will now reason, realize it needs to search the web, execute the search, and return the answer.
response = agent_with_internet.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

agent_message = response["messages"][-1].content
print(f"Agent: {agent_message}")

User: What is the latest spiderman movie?

Agent: Hey, kid—if you’re looking for the freshest web‑slinger on the big screen, it’s **“Spider‑Man: Brand New Day.”**  

- **Release:** slated for **July 31, 2026** (so you’ve got a little time to perfect your own suit).  
- **Cast:** Tom Holland is back as Peter Parker, with Zendaya, Sadie Sink, Jacob Batalon and a few surprise cameos.  
- **Director:** Destin Daniel Cretton (the guy who gave us the emotional punch of *Spider‑Man: No Way Home*).  
- **Where it fits:** It’s the fourth MCU‑Spider‑Man film and the 38th MCU entry overall, kicking off the final stretch of Phase 6.

So, if you’ve already binge‑watched *Across the Spider‑Verse* and *No Way Home*, get ready—Spidey’s swinging back with a brand‑new day, and the multiverse is about to get a little messier. Suit up, grab the popcorn, and enjoy the ride. 🚀🕸️


In [14]:
print(agent_message)

Hey, kid—if you’re looking for the freshest web‑slinger on the big screen, it’s **“Spider‑Man: Brand New Day.”**  

- **Release:** slated for **July 31, 2026** (so you’ve got a little time to perfect your own suit).  
- **Cast:** Tom Holland is back as Peter Parker, with Zendaya, Sadie Sink, Jacob Batalon and a few surprise cameos.  
- **Director:** Destin Daniel Cretton (the guy who gave us the emotional punch of *Spider‑Man: No Way Home*).  
- **Where it fits:** It’s the fourth MCU‑Spider‑Man film and the 38th MCU entry overall, kicking off the final stretch of Phase 6.

So, if you’ve already binge‑watched *Across the Spider‑Verse* and *No Way Home*, get ready—Spidey’s swinging back with a brand‑new day, and the multiverse is about to get a little messier. Suit up, grab the popcorn, and enjoy the ride. 🚀🕸️


# Moving Beyond Chat: Structured Output

So far, our Agent has been responding with standard conversational text. But if you are building software, you often don't want a conversational paragraph. You want **structured data** (like JSON or Python dictionaries) so your code can predictably use the results.

Imagine you are building an app that reads fantasy books and categorizes the lore. If we just ask the LLM to "extract the characters," it might write: *"Sure! The characters I found are Harry and Hermione. I hope that helps!"* — which is a nightmare to parse in code.

Instead, we can enforce a strict **Response Format** using Pydantic schemas. We tell the Agent: *"Do not talk to me. Only return a data object matching this exact structure."*

Let's look at an example using a short paragraph about Harry Potter. We want to extract Characters, Spells, and Magical Beasts into separate lists.

In [38]:
from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent

# 1. Define the exact structure we want the LLM to return
class HarryPotterEntities(BaseModel):
    characters: List[str] = Field(description="List of characters mentioned in the text")
    spells: List[str] = Field(description="List of magical spells cast")
    # magical_beasts: List[str] = Field(description="List of magical creatures and beasts")

# 2. Our sample text
lore_paragraph = """
Harry Potter and Hermione Granger ran through the Forbidden Forest. 
Suddenly, a massive Acromantula appeared from the shadows. 
Harry quickly shouted 'Expecto Patronum!' to ward off approaching Dementors, 
while Hermione cast 'Stupefy' at the giant spider. High above, a Hippogriff 
soared through the sky, completely unaware of the chaos below.
"""

# 3. Create the agent, passing our Pydantic model into `response_format`
extractor_agent =  create_agent(
    model=llm, 
    tools=[search], 
    response_format = HarryPotterEntities,
    system_prompt="You are my helpful Butler Alfred, seperate spells from characters"
)


In [42]:
# If this didnt work go with the below one
# # 4. Run the extraction
# print("Extracting entities...\n")
# response = extractor_agent.invoke(input =lore_paragraph)

# # Because we used a response format, the output is a predictable data object!
# extracted_data = response


In [40]:
llm_structured = llm.with_structured_output(HarryPotterEntities)

In [44]:
extracted_data = llm_structured.invoke(lore_paragraph)

In [45]:

print(f"Characters: {extracted_data.characters}")
print(f"Spells: {extracted_data.spells}")
# print(f"Magical Beasts: {extracted_data.magical_beasts}")

Characters: ['Harry Potter', 'Hermione Granger', 'Acromantula', 'Dementors', 'Hippogriff']
Spells: ['Expecto Patronum', 'Stupefy']


In [53]:
from langchain.tools import tool
from datetime import datetime

# 1. We use the @tool decorator to turn a standard Python function into an Agent Tool.
# The docstring ("""...""") is CRITICAL here—it's how the Agent reads and understands what the tool is used for!

@tool
def get_favorite_city() -> str:
    """Returns the favorite city of the user."""
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

# Let's test our new tool directly
print(f"Tool output directly: {get_current_time.invoke('')}")

Tool output directly: 2026-03-26 20:22:14


In [55]:
agent_with_internet = create_agent(
    model=llm, 
    tools=[search, get_favorite_city], 
    system_prompt="You are a helpful, assistant; The Alfred"
)

In [56]:
response = agent_with_internet.invoke(
    {"messages": [{"role": "user", "content": "what is my favorite city"}]}
)

In [59]:
response['messages'][-1].content

'I’m not seeing a city name in the information I have—just a timestamp. Could you let me know what your favorite city is? Once you share it, I’ll be happy to keep it in mind for any future recommendations or conversations.'

![](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/LangGraph/flow.png)

---
### Want to dive deeper? 
If you want to look under the hood of these models or explore other modalities (like Image-to-Text or Audio generation), check out these fantastic resources:
---
### Further Reading on Tools
You can give agents tools to read PDFs, query SQL databases, check your calendar, or even execute Python code! 
* **LangChain Built-in Tools:** [DuckDuckGo Integration Docs](https://docs.langchain.com/oss/python/integrations/tools/ddg)

* **Practical Implementation:** [Hugging Face NLP Course - Chapter 1](https://huggingface.co/learn/nlp-course/chapter1/3)
* **Visualizing the Architecture:** [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/)